# E-Commerce Sales & Customer Behaviour Analysis

**Dataset reference:** [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail)  
**Project scope:** descriptive EDA of supplied positive-sales and return/cancellation extracts.

> The ZIP supplied for this project contains prepared extracts and summary tables, not the original UCI workbook. This notebook analyses those provided files and explicitly checks reconciliation differences.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..")  # notebook is in notebooks/
DATA = ROOT / "data"
TABLES = DATA / "tables"
CHARTS = ROOT / "charts"
CHARTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")
sales = pd.read_csv(DATA / "positive_sales_transactions.csv", low_memory=False)
returns = pd.read_csv(DATA / "returns_cancellations_transactions.csv", low_memory=False)

sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"], errors="coerce")
for col in ["Quantity", "UnitPrice", "LineRevenue", "CustomerID"]:
    sales[col] = pd.to_numeric(sales[col], errors="coerce")
returns["InvoiceDate"] = pd.to_datetime(returns["InvoiceDate"], errors="coerce")

country = pd.read_csv(TABLES / "country_performance.csv")
monthly = pd.read_csv(TABLES / "monthly_sales.csv", parse_dates=["MonthStart"])
products = pd.read_csv(TABLES / "product_performance.csv")
rfm = pd.read_csv(TABLES / "rfm_segment_summary.csv")
kpi = pd.read_csv(TABLES / "kpi_summary.csv")
cleaning = pd.read_csv(TABLES / "data_cleaning_summary.csv")


## 1. Data understanding and quality checks

In [ ]:
print("Positive-sales shape:", sales.shape)
print("Returns/cancellations shape:", returns.shape)
display(sales.head())
display(sales.dtypes.to_frame("dtype"))
display(sales.describe(include="all").T)
display(sales.isna().sum().sort_values(ascending=False).to_frame("missing"))
print("Duplicate rows in supplied positive-sales extract:", sales.duplicated().sum())
print("Missing CustomerID share:", round(sales.CustomerID.isna().mean()*100, 2), "%")
display(cleaning)


## 2. KPI and reconciliation checks

In [ ]:
orders = (sales.groupby("InvoiceNo")
          .agg(OrderRevenue=("LineRevenue", "sum"),
               Units=("Quantity", "sum"),
               CustomerID=("CustomerID", "first"),
               InvoiceDate=("InvoiceDate", "min"))
          .reset_index())

print("Extract revenue:", sales.LineRevenue.sum())
print("Distinct invoices in positive-sales extract:", sales.InvoiceNo.nunique())
print("Mean order revenue in positive-sales extract:", orders.OrderRevenue.mean())
print("Median order revenue:", orders.OrderRevenue.median())
print("Identified customers:", sales.CustomerID.nunique())
print("Combined supplied sales + return lines:", len(sales) + len(returns))
display(kpi)
# Reconciliation differences are documented, not silently corrected.


## 3. Univariate analysis and visualizations

In [ ]:
numeric_cols = ["Quantity", "UnitPrice", "LineRevenue"]
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    upper = sales[col].quantile(.99)
    sns.histplot(sales[col].clip(upper=upper), bins=45)
    plt.title(f"Distribution of {col} (99th percentile cap for display)")
    plt.tight_layout()
    plt.show()

for col in numeric_cols:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=sales[col], showfliers=False)
    plt.title(f"Box plot: {col} (fliers hidden for readability)")
    plt.tight_layout()
    plt.show()


## 4. Bivariate and categorical analysis

In [ ]:
top_country = country.sort_values("Revenue", ascending=False).head(10)
plt.figure(figsize=(8, 5))
sns.barplot(data=top_country, y="Country", x="Revenue", color="#2864A5")
plt.title("Top 10 Countries by Revenue")
plt.tight_layout()
plt.show()

top_products = products.sort_values("Revenue", ascending=False).head(10)
plt.figure(figsize=(9, 5))
sns.barplot(data=top_products, y=top_products.Description.str.slice(0, 42), x="Revenue", color="#2864A5")
plt.title("Top 10 Products by Revenue")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.countplot(data=sales, x="Weekday", order=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
plt.xticks(rotation=25)
plt.title("Sales Lines by Weekday")
plt.tight_layout()
plt.show()

sample = sales.sample(min(12000, len(sales)), random_state=42)
plt.figure(figsize=(7, 5))
plt.scatter(sample.UnitPrice, sample.Quantity, alpha=.25, s=10)
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Unit price (£, log)"); plt.ylabel("Quantity (log)")
plt.title("Unit Price vs Quantity (sample)")
plt.tight_layout()
plt.show()


## 5. Multivariate, time-series and customer analysis

In [ ]:
corr = sales[["Quantity", "UnitPrice", "LineRevenue", "Hour"]].corr(numeric_only=True)
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="Blues", vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(monthly.MonthStart, monthly.Revenue, marker="o")
plt.title("Monthly Revenue")
plt.xlabel("Month"); plt.ylabel("Revenue (£)")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
sns.barplot(data=rfm, y="Segment", x="Customers", color="#2864A5")
plt.title("Customer Counts by RFM Segment")
plt.tight_layout()
plt.show()
display(rfm)


In [ ]:
## Export the full assignment visualization set (18 charts)

# Histograms (3)
for col, name in [("Quantity","quantity_histogram"),("UnitPrice","unit_price_histogram"),("LineRevenue","line_revenue_histogram")]:
    plt.figure(figsize=(8,4))
    sns.histplot(sales[col].clip(upper=sales[col].quantile(.99)), bins=45)
    plt.title(f"{col} distribution (99th percentile cap for display)")
    plt.tight_layout(); plt.savefig(CHARTS / f"{name}.png", dpi=180); plt.show()

# Box plots (3)
for col, name in [("Quantity","quantity_boxplot"),("UnitPrice","unit_price_boxplot"),("LineRevenue","line_revenue_boxplot")]:
    plt.figure(figsize=(8,3))
    sns.boxplot(x=sales[col], showfliers=False)
    plt.title(f"{col} box plot (fliers hidden for readability)")
    plt.tight_layout(); plt.savefig(CHARTS / f"{name}.png", dpi=180); plt.show()

# Bar charts (3)
for metric, title, name in [
    ("Revenue","Top products by revenue","products_revenue_bar"),
    ("UnitsSold","Top products by units sold","products_units_bar")]:
    top = products.sort_values(metric, ascending=False).head(10)
    plt.figure(figsize=(9,5))
    sns.barplot(data=top, y=top.Description.str.slice(0,42), x=metric)
    plt.title(title); plt.tight_layout()
    plt.savefig(CHARTS / f"{name}.png", dpi=180); plt.show()

top_countries = country.sort_values("Revenue", ascending=False).head(10)
plt.figure(figsize=(8,5))
sns.barplot(data=top_countries, y="Country", x="Revenue")
plt.title("Top countries by revenue"); plt.tight_layout()
plt.savefig(CHARTS / "countries_revenue_bar.png", dpi=180); plt.show()

# Count plots (2)
for col in ["Weekday","Hour"]:
    plt.figure(figsize=(8,4))
    sns.countplot(data=sales, x=col)
    plt.title(f"Sales lines by {col}")
    if col == "Weekday": plt.xticks(rotation=25)
    plt.tight_layout(); plt.savefig(CHARTS / f"{col.lower()}_countplot.png", dpi=180); plt.show()

# Scatter plots (2)
sample = sales.sample(min(12000, len(sales)), random_state=42)
plt.figure(figsize=(7,5)); plt.scatter(sample.UnitPrice, sample.Quantity, alpha=.25, s=10)
plt.xscale("log"); plt.yscale("log"); plt.xlabel("Unit price (£, log)"); plt.ylabel("Quantity (log)")
plt.title("Unit price vs quantity"); plt.tight_layout()
plt.savefig(CHARTS / "price_quantity_scatter.png", dpi=180); plt.show()

order_sample = orders.sample(min(8000, len(orders)), random_state=42)
plt.figure(figsize=(7,5)); plt.scatter(order_sample.Units, order_sample.OrderRevenue, alpha=.3, s=12)
plt.xscale("log"); plt.yscale("log"); plt.xlabel("Units per invoice (log)"); plt.ylabel("Order revenue (£, log)")
plt.title("Order units vs order revenue"); plt.tight_layout()
plt.savefig(CHARTS / "order_units_revenue_scatter.png", dpi=180); plt.show()

# Correlation heatmap (1)
corr = sales[["Quantity","UnitPrice","LineRevenue","Hour"]].corr(numeric_only=True)
plt.figure(figsize=(6,5)); sns.heatmap(corr, annot=True, cmap="Blues", vmin=-1, vmax=1)
plt.title("Correlation heatmap"); plt.tight_layout()
plt.savefig(CHARTS / "correlation_heatmap.png", dpi=180); plt.show()

# Time-series chart (1)
plt.figure(figsize=(9,4)); plt.plot(monthly.MonthStart, monthly.Revenue, marker="o")
plt.title("Monthly sales revenue"); plt.xlabel("Month"); plt.ylabel("Revenue (£)")
plt.xticks(rotation=35); plt.tight_layout()
plt.savefig(CHARTS / "monthly_revenue_timeseries.png", dpi=180); plt.show()

# Additional visuals (2): customer segments and weekday revenue
plt.figure(figsize=(9,4)); sns.barplot(data=rfm, y="Segment", x="Customers")
plt.title("Customer count by RFM segment"); plt.tight_layout()
plt.savefig(CHARTS / "rfm_segments.png", dpi=180); plt.show()

weekday_revenue = sales.groupby("Weekday").LineRevenue.sum().reindex(
    ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]).dropna()
plt.figure(figsize=(8,4)); weekday_revenue.plot(kind="bar")
plt.title("Revenue by weekday"); plt.ylabel("Revenue (£)")
plt.tight_layout(); plt.savefig(CHARTS / "weekday_revenue.png", dpi=180); plt.show()


## 6. Interpretation, recommendations and limitations

### Findings to report
- The UK contributes approximately 84.6% of revenue in the supplied country summary.
- November 2011 is the peak month in the supplied monthly table.
- The supplied RFM summary identifies a Champions segment and an At Risk segment with material revenue.
- Product leaders differ when ranked by revenue versus units sold.

### Recommendations
- Protect the core UK market while testing selected international opportunities.
- Prepare stock and fulfilment capacity ahead of the observed peak, then validate seasonality using more years.
- Run measured retention and reactivation experiments for high-value and at-risk segments.
- Improve CustomerID capture and reconcile the supplied extracts against the original source before financial reporting.

### Limitations
- Original UCI workbook and preprocessing script were not included in the supplied ZIP.
- KPI and extract invoice counts differ by two; the cleaned master row count also differs from the combined supplied extracts.
- CustomerID is missing on approximately 25.2% of positive-sales lines.
- The period is about one year and December 2011 is partial.
- Correlation and descriptive EDA do not establish causation.
